In [22]:
import pandas as pd
import numpy as np

In [23]:
import os

os.chdir(r"C:\Users\hp\Documents\Oasis Infobyte Tasks or Projects Documents\Task 3\Task 3 Documents\extracted csv file")

print(os.getcwd())

C:\Users\hp\Documents\Oasis Infobyte Tasks or Projects Documents\Task 3\Task 3 Documents\extracted csv file


In [24]:
# Preserve the original dataset for before-vs-after comparison
df_before = df.copy()

In [25]:
print(os.listdir())

['cleaned_cafe_sales.csv', 'Data_Cleaning.ipynb', 'dirty_cafe_sales.csv']


In [26]:
df = pd.read_csv("dirty_cafe_sales.csv")

In [27]:
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [29]:
df.shape

(10000, 8)

In [30]:
df.columns.tolist()

['Transaction ID',
 'Item',
 'Quantity',
 'Price Per Unit',
 'Total Spent',
 'Payment Method',
 'Location',
 'Transaction Date']

In [31]:
df.isnull().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

In [32]:
df.isnull().sum().sort_values(ascending=False)

Location            3265
Payment Method      2579
Item                 333
Price Per Unit       179
Total Spent          173
Transaction Date     159
Quantity             138
Transaction ID         0
dtype: int64

In [33]:
df.duplicated().sum()

np.int64(0)

In [34]:
df[df.duplicated()]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date


In [35]:
for column in df.columns:
    print(f"\n--- {column} ---")
    print(df[column].unique()[:20])


--- Transaction ID ---
['TXN_1961373' 'TXN_4977031' 'TXN_4271903' 'TXN_7034554' 'TXN_3160411'
 'TXN_2602893' 'TXN_4433211' 'TXN_6699534' 'TXN_4717867' 'TXN_2064365'
 'TXN_2548360' 'TXN_3051279' 'TXN_7619095' 'TXN_9437049' 'TXN_8915701'
 'TXN_2847255' 'TXN_3765707' 'TXN_6769710' 'TXN_8876618' 'TXN_3709394']

--- Item ---
['Coffee' 'Cake' 'Cookie' 'Salad' 'Smoothie' 'UNKNOWN' 'Sandwich' nan
 'ERROR' 'Juice' 'Tea']

--- Quantity ---
['2' '4' '5' '3' '1' 'ERROR' 'UNKNOWN' nan]

--- Price Per Unit ---
['2.0' '3.0' '1.0' '5.0' '4.0' '1.5' nan 'ERROR' 'UNKNOWN']

--- Total Spent ---
['4.0' '12.0' 'ERROR' '10.0' '20.0' '9.0' '16.0' '15.0' '25.0' '8.0' '5.0'
 '3.0' '6.0' nan 'UNKNOWN' '2.0' '1.0' '7.5' '4.5' '1.5']

--- Payment Method ---
['Credit Card' 'Cash' 'UNKNOWN' 'Digital Wallet' 'ERROR' nan]

--- Location ---
['Takeaway' 'In-store' 'UNKNOWN' nan 'ERROR']

--- Transaction Date ---
['2023-09-08' '2023-05-16' '2023-07-19' '2023-04-27' '2023-06-11'
 '2023-03-31' '2023-10-06' '2023-10-28' '

1. Data Quality Report — Before Cleaning



Before cleaning the dataset, a data quality assessment was performed to identify missing values, duplicate records, incorrect data types, and invalid or inconsistent values. This report establishes the baseline condition of the dataset before any transformations are applied.

In [36]:
quality_report = pd.DataFrame({
    "Column": df.columns,
    "Missing Values": df.isnull().sum().values,
    "Missing %": (df.isnull().sum().values / len(df) * 100).round(2),
    "Data Type": df.dtypes.astype(str).values,
    "Unique Values": df.nunique(dropna=True).values
})

quality_report

,Column,Missing Values,Missing %,Data Type,Unique Values
0,Transaction ID,0,0.00,object,10000
1,Item,333,3.33,object,10
2,Quantity,138,1.38,object,7
3,Price Per Unit,179,1.79,object,8
4,Total Spent,173,1.73,object,19
5,Payment Method,2579,25.79,object,5
6,Location,3265,32.65,object,4
7,Transaction Date,159,1.59,object,367


In [37]:
invalid_values = ["ERROR", "UNKNOWN"]

invalid_report = pd.DataFrame({
    "Column": df.columns,
    "ERROR Count": [(df[col] == "ERROR").sum() for col in df.columns],
    "UNKNOWN Count": [(df[col] == "UNKNOWN").sum() for col in df.columns]
})

invalid_report

,Column,ERROR Count,UNKNOWN Count
0,Transaction ID,0,0
1,Item,292,344
2,Quantity,170,171
3,Price Per Unit,190,164
4,Total Spent,164,165
5,Payment Method,306,293
6,Location,358,338
7,Transaction Date,142,159


In [38]:
for column in ["Quantity", "Price Per Unit", "Total Spent"]:
    numeric_values = pd.to_numeric(df[column], errors="coerce")
    
    print(f"\n--- {column} ---")
    print(f"Minimum: {numeric_values.min()}")
    print(f"Maximum: {numeric_values.max()}")
    print(f"Mean: {numeric_values.mean():.2f}")
    print(f"Median: {numeric_values.median():.2f}")


--- Quantity ---
Minimum: 1.0
Maximum: 5.0
Mean: 3.03
Median: 3.00

--- Price Per Unit ---
Minimum: 1.0
Maximum: 5.0
Mean: 2.95
Median: 3.00

--- Total Spent ---
Minimum: 1.0
Maximum: 25.0
Mean: 8.92
Median: 8.00


## 2. Missing Data Handling

The dataset contains both standard missing values (`NaN`) and placeholder values such as `ERROR` and `UNKNOWN`. These placeholders were treated as missing information because they do not represent valid observations.

The cleaning strategy was selected according to the nature of each column:

- **Item:** Mode imputation, because it is a categorical variable and the most frequent valid item is a reasonable replacement.
- **Quantity:** Median imputation, because it is a numerical variable and the median is less sensitive to potential outliers.
- **Price Per Unit:** Median imputation, because it is numerical and the median provides a robust estimate.
- **Total Spent:** Median imputation, because it is numerical and may contain extreme values.
- **Payment Method:** Mode imputation, because it is categorical.
- **Location:** Mode imputation, because it is categorical.
- **Transaction Date:** The invalid/missing dates will be converted to missing values and handled using the mode date, since this dataset requires a complete transaction-date field for analysis.
- **Transaction ID:** No imputation is required because this column contains no missing values.

In [39]:
# Replace invalid placeholder values with NaN
df = df.replace(["ERROR", "UNKNOWN"], np.nan)

In [40]:
df.isnull().sum()

Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

In [41]:
numeric_columns = ["Quantity", "Price Per Unit", "Total Spent"]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

In [42]:
df[numeric_columns].dtypes

Quantity          float64
Price Per Unit    float64
Total Spent       float64
dtype: object

In [43]:
df["Transaction Date"] = pd.to_datetime(
    df["Transaction Date"],
    errors="coerce"
)

In [44]:
df["Transaction Date"].dtype

dtype('<M8[ns]')

In [45]:
# Convert numerical columns to numeric values
numeric_columns = ["Quantity", "Price Per Unit", "Total Spent"]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

In [46]:

# Reconstruct Total Spent when Quantity and Price Per Unit are available
mask_total = (
    df["Total Spent"].isna()
    & df["Quantity"].notna()
    & df["Price Per Unit"].notna()
)

df.loc[mask_total, "Total Spent"] = (
    df.loc[mask_total, "Quantity"] *
    df.loc[mask_total, "Price Per Unit"]
)

In [47]:
# Reconstruct Quantity when Price Per Unit and Total Spent are available
mask_quantity = (
    df["Quantity"].isna()
    & df["Price Per Unit"].notna()
    & df["Total Spent"].notna()
    & (df["Price Per Unit"] != 0)
)

df.loc[mask_quantity, "Quantity"] = (
    df.loc[mask_quantity, "Total Spent"] /
    df.loc[mask_quantity, "Price Per Unit"]
)

In [48]:
# Reconstruct Price Per Unit when Quantity and Total Spent are available
mask_price = (
    df["Price Per Unit"].isna()
    & df["Quantity"].notna()
    & df["Total Spent"].notna()
    & (df["Quantity"] != 0)
)

df.loc[mask_price, "Price Per Unit"] = (
    df.loc[mask_price, "Total Spent"] /
    df.loc[mask_price, "Quantity"]
)

In [49]:
for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

In [50]:
# Ensure Total Spent is consistent with Quantity × Price Per Unit
df["Total Spent"] = df["Quantity"] * df["Price Per Unit"]

In [51]:
calculated_total = df["Quantity"] * df["Price Per Unit"]

inconsistent_rows = ~np.isclose(
    calculated_total,
    df["Total Spent"],
    rtol=0.01,
    atol=0.01
)

print("Inconsistent rows:", inconsistent_rows.sum())

Inconsistent rows: 0


### Numerical Consistency

The numerical fields were checked for internal consistency because `Total Spent` is logically derived from `Quantity × Price Per Unit`. Where sufficient information was available, missing numerical values were reconstructed using this relationship. Remaining missing numerical values were imputed using the median.

After imputation, `Total Spent` was recalculated from `Quantity × Price Per Unit` to ensure that the three numerical fields remained internally consistent.

In [52]:
calculated_total = df["Quantity"] * df["Price Per Unit"]

inconsistent_rows = ~np.isclose(
    calculated_total,
    df["Total Spent"],
    rtol=0.01,
    atol=0.01
)

print("Inconsistent Quantity × Price vs Total Spent rows:",
      inconsistent_rows.sum())

Inconsistent Quantity × Price vs Total Spent rows: 0


In [53]:
df[numeric_columns].isnull().sum()

Quantity          0
Price Per Unit    0
Total Spent       0
dtype: int64

In [54]:
categorical_columns = ["Item", "Payment Method", "Location"]

for column in categorical_columns:
    df[column] = df[column].fillna(df[column].mode()[0])

In [55]:
df[categorical_columns].isnull().sum()

Item              0
Payment Method    0
Location          0
dtype: int64

In [56]:
df["Transaction Date"] = df["Transaction Date"].fillna(
    df["Transaction Date"].mode()[0]
)

In [57]:
df.isnull().sum()

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

## 3. Duplicate Removal

Duplicate rows were checked using all columns. No exact duplicate rows were found in the original dataset, so no rows were removed during this step.

In [58]:
duplicate_count = df.duplicated().sum()

print(f"Number of duplicate rows: {duplicate_count}")

Number of duplicate rows: 0


## 4. Standardization

Categorical values were standardized by replacing invalid placeholder values such as `ERROR` and `UNKNOWN` with missing values and subsequently imputing them using the most frequent valid category. This ensures that categorical columns contain consistent, analysis-ready values.

In [59]:
categorical_columns = ["Item", "Payment Method", "Location"]

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(df[column].value_counts())


--- Item ---
Item
Juice       2140
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
Name: count, dtype: int64

--- Payment Method ---
Payment Method
Digital Wallet    5469
Credit Card       2273
Cash              2258
Name: count, dtype: int64

--- Location ---
Location
Takeaway    6983
In-store    3017
Name: count, dtype: int64


### Date Standardization

The `Transaction Date` column was converted from a text/object data type to the standard pandas `datetime64[ns]` format. Invalid date entries were converted to missing values and subsequently imputed using the mode date.

In [60]:
df["Transaction Date"].dtype

dtype('<M8[ns]')

## 5. Outlier Detection

The Interquartile Range (IQR) method was used to identify potential outliers in the numerical columns. The IQR method identifies observations below Q1 − 1.5 × IQR or above Q3 + 1.5 × IQR as potential outliers.

Potential outliers were evaluated before deciding whether they should be removed, capped, or retained.

In [61]:
numeric_columns = ["Quantity", "Price Per Unit", "Total Spent"]

outlier_report = []

for column in numeric_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[
        (df[column] < lower_bound) | 
        (df[column] > upper_bound)
    ]
    
    outlier_report.append({
        "Column": column,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": len(outliers)
    })

outlier_report = pd.DataFrame(outlier_report)

outlier_report

,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count
0,Quantity,2.0,4.0,2.0,-1.0,7.0,0
1,Price Per Unit,2.0,4.0,2.0,-1.0,7.0,0
2,Total Spent,4.0,12.0,8.0,-8.0,24.0,267


### Evaluation of Total Spent Outliers

The IQR method identified 259 potential outliers in the `Total Spent` column. These observations were inspected before deciding how to handle them. Since unusually high transaction amounts can represent legitimate purchases rather than data-entry errors, the outliers will be retained unless they are shown to be invalid.

In [62]:
total_spent_outliers = df[df["Total Spent"] > 24]

total_spent_outliers[[
    "Transaction ID",
    "Item",
    "Quantity",
    "Price Per Unit",
    "Total Spent"
]].head(20)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent
10,TXN_2548360,Salad,5.0,5.0,25.0
51,TXN_6342161,Salad,5.0,5.0,25.0
52,TXN_8914892,Juice,5.0,5.0,25.0
96,TXN_5220895,Salad,5.0,5.0,25.0
100,TXN_9517146,Juice,5.0,5.0,25.0
150,TXN_8687151,Salad,5.0,5.0,25.0
157,TXN_4283157,Salad,5.0,5.0,25.0
177,TXN_1896955,Salad,5.0,5.0,25.0
214,TXN_8693704,Salad,5.0,5.0,25.0
330,TXN_5523450,Salad,5.0,5.0,25.0


In [63]:
total_spent_outliers["Total Spent"].describe()

count    267.0
mean      25.0
std        0.0
min       25.0
25%       25.0
50%       25.0
75%       25.0
max       25.0
Name: Total Spent, dtype: float64

### Outlier Handling Decision

The IQR method identified 259 potential outliers in the `Total Spent` column. All identified values were exactly 25.0, which is only slightly above the calculated upper IQR boundary of 24.0.

These values are plausible transaction amounts and do not indicate obvious data-entry errors. Therefore, the outliers were retained rather than removed or capped to avoid altering legitimate sales information.

In [64]:
print("Outliers detected in Total Spent:", len(total_spent_outliers))
print("Outlier value(s):", total_spent_outliers["Total Spent"].unique())
print("Decision: Retain legitimate outliers.")

Outliers detected in Total Spent: 267
Outlier value(s): [25.]
Decision: Retain legitimate outliers.


## 6. Data Type Correction

Data types were corrected to ensure that each column is stored in an appropriate format for analysis:

- `Transaction ID` is stored as a string because it is an identifier rather than a numerical measurement.
- `Item`, `Payment Method`, and `Location` are stored as categorical/text values.
- `Quantity` is stored as an integer because it represents the number of items purchased.
- `Price Per Unit` and `Total Spent` are stored as floating-point numbers because they represent monetary values.
- `Transaction Date` is stored as a datetime value.

In [65]:
df["Transaction ID"] = df["Transaction ID"].astype("string")

df["Item"] = df["Item"].astype("string")
df["Payment Method"] = df["Payment Method"].astype("string")
df["Location"] = df["Location"].astype("string")

df["Quantity"] = df["Quantity"].astype("int64")
df["Price Per Unit"] = df["Price Per Unit"].astype("float64")
df["Total Spent"] = df["Total Spent"].astype("float64")

df["Transaction Date"] = pd.to_datetime(df["Transaction Date"])

In [66]:
df.dtypes

Transaction ID      string[python]
Item                string[python]
Quantity                     int64
Price Per Unit             float64
Total Spent                float64
Payment Method      string[python]
Location            string[python]
Transaction Date    datetime64[ns]
dtype: object

## 7. Before vs. After Cleaning Summary

The following summary compares the key data-quality indicators before and after cleaning. It shows the changes in missing values, duplicate rows, row count, and data-type accuracy.

In [67]:
# Load the original dataset only for the before-cleaning comparison
df_before = pd.read_csv("dirty_cafe_sales.csv")

print("Original dataset loaded:", df_before.shape)
print("Original total nulls:", df_before.isnull().sum().sum())

Original dataset loaded: (10000, 8)
Original total nulls: 6826


In [68]:
# Before-cleaning metrics
before_missing = int(df_before.isnull().sum().sum())
before_duplicates = int(df_before.duplicated().sum())
before_rows = len(df_before)
before_correct_dtypes = 0

# After-cleaning metrics
after_missing = int(df.isnull().sum().sum())
after_duplicates = int(df.duplicated().sum())
after_rows = len(df)

# Expected data types
expected_dtypes = {
    "Transaction ID": "string",
    "Item": "string",
    "Quantity": "int64",
    "Price Per Unit": "float64",
    "Total Spent": "float64",
    "Payment Method": "string",
    "Location": "string",
    "Transaction Date": "datetime64[ns]"
}

# Check final data-type accuracy
after_correct_dtypes = sum(
    str(df[column].dtype) == dtype
    for column, dtype in expected_dtypes.items()
)

# Create summary table
summary = pd.DataFrame({
    "Metric": [
        "Total Missing Values",
        "Duplicate Rows",
        "Row Count",
        "Correct Data Types"
    ],
    "Before Cleaning": [
        before_missing,
        before_duplicates,
        before_rows,
        f"{before_correct_dtypes}/8"
    ],
    "After Cleaning": [
        after_missing,
        after_duplicates,
        after_rows,
        f"{after_correct_dtypes}/8"
    ]
})

summary

,Metric,Before Cleaning,After Cleaning
0,Total Missing Values,6826,0
1,Duplicate Rows,0,0
2,Row Count,10000,10000
3,Correct Data Types,0/8,8/8


## 8. Final Data Quality Validation

The cleaned dataset was validated to confirm that missing values, duplicate rows, and incorrect data types were addressed before exporting the final dataset.

In [69]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

print("\nDataset shape:")
print(df.shape)

Missing values:
Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

Duplicate rows:
0

Data types:
Transaction ID      string[python]
Item                string[python]
Quantity                     int64
Price Per Unit             float64
Total Spent                float64
Payment Method      string[python]
Location            string[python]
Transaction Date    datetime64[ns]
dtype: object

Dataset shape:
(10000, 8)


In [70]:
df.head(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,Digital Wallet,Takeaway,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,Takeaway,2023-03-31
6,TXN_4433211,Juice,3,3.0,9.0,Digital Wallet,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,Takeaway,2023-10-28
8,TXN_4717867,Juice,5,3.0,15.0,Digital Wallet,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,Digital Wallet,In-store,2023-12-31


In [71]:
df.tail(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
9990,TXN_1538510,Coffee,5,2.0,10.0,Digital Wallet,Takeaway,2023-05-22
9991,TXN_3897619,Sandwich,3,4.0,12.0,Cash,Takeaway,2023-02-24
9992,TXN_2739140,Smoothie,4,4.0,16.0,Digital Wallet,In-store,2023-07-05
9993,TXN_4766549,Smoothie,2,4.0,8.0,Cash,Takeaway,2023-10-20
9994,TXN_7851634,Juice,4,4.0,16.0,Digital Wallet,Takeaway,2023-01-08
9995,TXN_7672686,Coffee,2,2.0,4.0,Digital Wallet,Takeaway,2023-08-30
9996,TXN_9659401,Juice,3,1.0,3.0,Digital Wallet,Takeaway,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,Takeaway,2023-03-02
9998,TXN_7695629,Cookie,3,1.0,3.0,Digital Wallet,Takeaway,2023-12-02
9999,TXN_6170729,Sandwich,3,4.0,12.0,Cash,In-store,2023-11-07


## 9. Export Cleaned Dataset

The cleaned and validated dataset was exported as a new CSV file so that the original raw dataset remains unchanged.

In [72]:
df.to_csv("cleaned_cafe_sales.csv", index=False)
print("Cleaned CSV updated successfully.")

Cleaned CSV updated successfully.


In [73]:
import os

print(os.path.exists("cleaned_cafe_sales.csv"))

True


In [74]:
print(os.path.getsize("cleaned_cafe_sales.csv"), "bytes")

620151 bytes


In [75]:
cleaned_check = pd.read_csv("cleaned_cafe_sales.csv")

print("Shape:", cleaned_check.shape)
print("Missing values:", cleaned_check.isnull().sum().sum())
print("Duplicate rows:", cleaned_check.duplicated().sum())

Shape: (10000, 8)
Missing values: 0
Duplicate rows: 0
